# 15 Kaggle - DNABERT-2 REF/ALT Pair Fine-Tuning for ClinVar SNV

Standalone Kaggle notebook dùng cùng data pipeline với notebook 14:

```text
Raw ClinVar variant_summary + GRCh38 FASTA
-> fixed_refseq filter có LastEvaluated
-> sequence parquet 601bp: ref_seq + alt_seq
-> pseudo-temporal x genome-block split
-> DNABERT-2 REF/ALT pair fine-tuning
```

Ràng buộc:

- Input chỉ đọc từ `/kaggle/input`.
- Output/cache/model chỉ ghi vào `/kaggle/working`.
- Không dùng dense NPY; cache dataset là parquet sequence.
- `time_only` mặc định không áp coordinate purge 5kb để tránh cắt quá đà future rows.


In [ ]:
from pathlib import Path
import json
import os
import random
import time
import hashlib
import math
import traceback
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm
from IPython.display import display

try:
    from pyfaidx import Fasta
except ImportError as exc:
    raise ImportError("pyfaidx is required. On Kaggle, install/add pyfaidx before running this notebook.") from exc

# Kaggle internet can still fail on Hugging Face Xet/CAS downloads; use normal HTTP downloads.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")

try:
    from transformers import AutoConfig, AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
except ImportError as exc:
    raise ImportError("transformers is required. On Kaggle, enable internet or install/add transformers before running this notebook.") from exc

IS_KAGGLE = Path("/kaggle/input").exists()
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path.cwd() / "Data"
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd() / "kaggle_work_dnabert2"
PROCESSED_DIR = WORK_DIR / "processed"
MODEL_DIR = WORK_DIR / "models"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Set these only if Kaggle mounted files under unusual names.
KAGGLE_INPUT_DIR_OVERRIDE = ""
VARIANT_SUMMARY_PATH_OVERRIDE = ""
FASTA_PATH_OVERRIDE = ""

LENGTH = 601
FLANK = LENGTH // 2
DATASET_TAG = "fixed_refseq_temporal_seqpair"
LABEL_MODE = "all"  # all | definitive_only

SPLIT_MODES = ["space_time_block"]  # space_only_matched | time_only | space_time_block | genome_block
TEMPORAL_TRAIN_MAX_YEAR = 2022
TEMPORAL_VAL_YEAR = 2023
TEMPORAL_TEST_MIN_YEAR = 2024
DROP_MISSING_LAST_EVALUATED = True
EXACT_VARIANT_PURGE = True
PURGE_DISTANCE_BP = 5000
APPLY_COORDINATE_PURGE_TO_TIME_ONLY = False
GENOME_BLOCK_SIZE_BP = 1_000_000
SEQUENCE_PURGE_MODE = "exact_ref"  # none | exact_ref | exact_tensor
MATCH_SPACE_ONLY_TRAIN_SIZE = True
SPACE_ONLY_TARGET_TRAIN_ROWS = None

MODEL_NAME_OR_PATH = "zhihan1996/DNABERT-2-117M"
MODEL_LOCAL_DIR_OVERRIDE = ""  # e.g. /kaggle/input/dnabert2-117m/DNABERT-2-117M
TRUST_REMOTE_CODE = True
FORCE_PYTORCH_ATTENTION = True
PYTORCH_ATTENTION_DROPOUT = 1e-12
LOW_CPU_MEM_USAGE = False  # DNABERT-2 remote code builds ALiBi tensors during __init__, so meta-device loading can fail.
MAX_TOKEN_LENGTH = 512
FINETUNE_MODE = "last_layers"  # head_only | last_layers | full
UNFREEZE_LAST_N_LAYERS = 4
DROPOUT = 0.15

BATCH_SIZE = 8 if IS_KAGGLE else 2
GRAD_ACCUM_STEPS = 8
EPOCHS = 2
ENCODER_LR = 2e-5
CLASSIFIER_LR = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.06
GRAD_CLIP_NORM = 1.0
IMBALANCE_STRATEGY = "weighted_sampler"  # none | weighted_sampler | pos_weight
USE_AMP = True
NUM_WORKERS = 2 if IS_KAGGLE else 0
PREFETCH_FACTOR = 2
PERSISTENT_WORKERS = NUM_WORKERS > 0
RANDOM_STATE = 42

RUN_FILTER_RAW = True
RUN_BUILD_DATASET = True
RUN_SMOKE_TEST = True
RUN_MINI_TRAIN = False
RUN_FULL_TRAIN = True
RUN_NEGATIVE_CONTROLS = False
NEGATIVE_CONTROL_MAX_ROWS = 20_000  # None = toàn bộ test set.
NEGATIVE_CONTROL_BATCH_SIZE = BATCH_SIZE
DEBUG_MAX_TRAIN_ROWS = 1024
DEBUG_MAX_VAL_ROWS = 256
DEBUG_MAX_TEST_ROWS = 256

SAFE_BASE = (
    f"dnabert2_ref_alt_pair_maxlen{MAX_TOKEN_LENGTH}_{LENGTH}_{DATASET_TAG}_"
    f"{FINETUNE_MODE}{UNFREEZE_LAST_N_LAYERS}_imbalance_{IMBALANCE_STRATEGY}"
)

print("IS_KAGGLE:", IS_KAGGLE)
print("INPUT_ROOT:", INPUT_ROOT)
print("WORK_DIR:", WORK_DIR)
print("SPLIT_MODES:", SPLIT_MODES)
print("MODEL_NAME_OR_PATH:", MODEL_NAME_OR_PATH)


In [ ]:
CHROMOSOME_TO_REFSEQ = {
    "1": "NC_000001.11", "2": "NC_000002.12", "3": "NC_000003.12", "4": "NC_000004.12",
    "5": "NC_000005.10", "6": "NC_000006.12", "7": "NC_000007.14", "8": "NC_000008.11",
    "9": "NC_000009.12", "10": "NC_000010.11", "11": "NC_000011.10", "12": "NC_000012.12",
    "13": "NC_000013.11", "14": "NC_000014.9", "15": "NC_000015.10", "16": "NC_000016.10",
    "17": "NC_000017.11", "18": "NC_000018.10", "19": "NC_000019.10", "20": "NC_000020.11",
    "21": "NC_000021.9", "22": "NC_000022.11", "X": "NC_000023.11", "Y": "NC_000024.10",
    "MT": "NC_012920.1", "M": "NC_012920.1",
}
POSITIVE_LABELS = {"Pathogenic", "Likely pathogenic"}
NEGATIVE_LABELS = {"Benign", "Likely benign"}
TRUSTED_REVIEW_PATTERN = "criteria provided|reviewed by expert panel|practice guideline"
LOW_CONFIDENCE_REVIEW = "no assertion criteria provided"
BASE_TO_INDEX = np.full(256, 255, dtype=np.uint8)
for _idx, _base in enumerate(b"ACGT"):
    BASE_TO_INDEX[_base] = _idx


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def discover_variant_summary():
    if VARIANT_SUMMARY_PATH_OVERRIDE:
        p = Path(VARIANT_SUMMARY_PATH_OVERRIDE)
        if p.exists():
            return p
    search_root = Path(KAGGLE_INPUT_DIR_OVERRIDE) if KAGGLE_INPUT_DIR_OVERRIDE else INPUT_ROOT
    candidates = sorted(search_root.rglob("variant_summary.txt")) if search_root.exists() else []
    if candidates:
        return candidates[0]
    local = Path.cwd() / "Data" / "variant_summary.txt"
    if local.exists():
        return local
    raise FileNotFoundError("Cannot find variant_summary.txt in /kaggle/input or local Data/.")


def discover_fasta():
    if FASTA_PATH_OVERRIDE:
        p = Path(FASTA_PATH_OVERRIDE)
        if p.exists():
            return p
    search_root = Path(KAGGLE_INPUT_DIR_OVERRIDE) if KAGGLE_INPUT_DIR_OVERRIDE else INPUT_ROOT
    for pattern in ["*GRCh38_genomic.fna", "*GRCh38*.fna", "*.fna"]:
        candidates = sorted(search_root.rglob(pattern)) if search_root.exists() else []
        if candidates:
            return candidates[0]
    local = Path.cwd() / "Data" / "ncbi_dataset" / "ncbi_dataset" / "data" / "GCF_000001405.26" / "GCF_000001405.26_GRCh38_genomic.fna"
    if local.exists():
        return local
    raise FileNotFoundError("Cannot find GRCh38 FASTA in /kaggle/input or local Data/.")


def split_clinical_significance(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("/") if part.strip()]


def map_clean_label(value):
    labels = split_clinical_significance(value)
    label_set = set(labels)
    if not labels:
        return None
    if label_set <= POSITIVE_LABELS:
        return 1
    if label_set <= NEGATIVE_LABELS:
        return 0
    return None


def normalize_chromosome(chromosome):
    chrom = str(chromosome).strip()
    if chrom.lower().startswith("chr"):
        chrom = chrom[3:]
    if chrom in {"M", "MT", "m", "mt"}:
        return "MT"
    return chrom.upper()


def clean_base(value):
    return str(value).strip().upper()


def choose_chromosome_fasta(row, fasta_names):
    chrom = normalize_chromosome(row["Chromosome"])
    accession = str(row.get("ChromosomeAccession", "")).strip()
    candidates = [accession, str(row["Chromosome"]).strip(), chrom, f"chr{chrom}", CHROMOSOME_TO_REFSEQ.get(chrom, "")]
    for candidate in candidates:
        if candidate and candidate in fasta_names:
            return candidate
    return ""

set_seed(RANDOM_STATE)
VARIANT_SUMMARY_PATH = discover_variant_summary()
FASTA_PATH = discover_fasta()
FILTERED_PATH = PROCESSED_DIR / "clinvar_filtered_step8_fixed_refseq_temporal.parquet"
SEQUENCE_DATASET_PATH = PROCESSED_DIR / f"clinvar_ref_alt_pair_{LENGTH}_{DATASET_TAG}.parquet"

print("VARIANT_SUMMARY:", VARIANT_SUMMARY_PATH, VARIANT_SUMMARY_PATH.exists())
print("FASTA:", FASTA_PATH, FASTA_PATH.exists())
print("FILTERED_PATH:", FILTERED_PATH)
print("SEQUENCE_DATASET_PATH:", SEQUENCE_DATASET_PATH)


In [ ]:
def filter_raw_clinvar():
    if FILTERED_PATH.exists() and not RUN_FILTER_RAW:
        print("Use existing filtered parquet:", FILTERED_PATH)
        return pd.read_parquet(FILTERED_PATH)

    genome = Fasta(str(FASTA_PATH), as_raw=True, sequence_always_upper=True, rebuild=False)
    fasta_names = set(genome.keys())
    missing_primary = {chrom: acc for chrom, acc in CHROMOSOME_TO_REFSEQ.items() if acc not in fasta_names}
    if missing_primary:
        raise RuntimeError(f"Missing primary RefSeq accessions in FASTA: {missing_primary}")

    usecols = [
        "Type", "GeneSymbol", "ClinicalSignificance", "LastEvaluated", "Assembly",
        "ChromosomeAccession", "Chromosome", "ReviewStatus", "PositionVCF",
        "ReferenceAlleleVCF", "AlternateAlleleVCF",
    ]
    counters = Counter()
    kept_chunks = []
    reader = pd.read_csv(VARIANT_SUMMARY_PATH, sep="\t", usecols=usecols, chunksize=500_000, low_memory=False)

    for chunk in tqdm(reader, desc="Filter ClinVar chunks"):
        counters["raw_rows"] += len(chunk)
        chunk = chunk.copy()
        chunk["Y"] = chunk["ClinicalSignificance"].map(map_clean_label)
        mask = chunk["Y"].notna()
        counters["labeled_rows"] += int(mask.sum())
        mask &= chunk["Assembly"].eq("GRCh38")
        counters["grch38_labeled_rows"] += int(mask.sum())
        mask &= chunk["Type"].eq("single nucleotide variant")
        counters["snv_rows"] += int(mask.sum())
        review_status = chunk["ReviewStatus"].fillna("").astype(str)
        trusted_review = review_status.str.contains(TRUSTED_REVIEW_PATTERN, case=False, regex=True)
        low_confidence = review_status.str.fullmatch(LOW_CONFIDENCE_REVIEW, case=False)
        mask &= trusted_review & ~low_confidence
        counters["trusted_rows"] += int(mask.sum())
        last_eval = pd.to_datetime(chunk["LastEvaluated"], errors="coerce")
        mask &= last_eval.notna()
        counters["last_evaluated_rows"] += int(mask.sum())
        ref = chunk["ReferenceAlleleVCF"].map(clean_base)
        alt = chunk["AlternateAlleleVCF"].map(clean_base)
        position = pd.to_numeric(chunk["PositionVCF"], errors="coerce")
        mask &= ref.str.fullmatch("[ACGT]") & alt.str.fullmatch("[ACGT]") & position.notna()
        counters["basic_allele_position_rows"] += int(mask.sum())
        out = chunk.loc[mask].copy()
        if out.empty:
            continue
        out["ReferenceAlleleVCF"] = ref.loc[mask].to_numpy()
        out["AlternateAlleleVCF"] = alt.loc[mask].to_numpy()
        out["PositionVCF"] = position.loc[mask].astype(np.int64).to_numpy()
        out["Y"] = out["Y"].astype(np.int8)
        out["Chromosome"] = out["Chromosome"].map(normalize_chromosome)
        out["ChromosomeFASTA"] = out.apply(choose_chromosome_fasta, axis=1, fasta_names=fasta_names)
        out = out[out["ChromosomeFASTA"].isin(fasta_names)].copy()
        counters["chromosome_fasta_rows"] += len(out)
        if out.empty:
            continue
        reference_bases = []
        matches = []
        for row in out.itertuples(index=False):
            base = str(genome[str(row.ChromosomeFASTA)][int(row.PositionVCF) - 1:int(row.PositionVCF)]).upper()
            reference_bases.append(base)
            matches.append(base == row.ReferenceAlleleVCF)
        out["ReferenceBaseGRCh38"] = reference_bases
        out["ReferenceAlleleMatchesGRCh38"] = matches
        out = out[out["ReferenceAlleleMatchesGRCh38"]].copy()
        counters["ref_match_rows"] += len(out)
        if out.empty:
            continue
        ordered_cols = [
            "Type", "GeneSymbol", "ClinicalSignificance", "LastEvaluated", "Assembly", "Chromosome",
            "ReviewStatus", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF", "Y",
            "ChromosomeFASTA", "ReferenceBaseGRCh38", "ReferenceAlleleMatchesGRCh38",
        ]
        kept_chunks.append(out[ordered_cols])

    if not kept_chunks:
        raise RuntimeError("No rows passed filtering.")
    filtered = pd.concat(kept_chunks, ignore_index=True)
    filtered.to_parquet(FILTERED_PATH, index=False)
    print("Counters:", dict(counters))
    print("Saved:", FILTERED_PATH)
    print("Shape:", filtered.shape)
    return filtered

filtered = filter_raw_clinvar()
display(filtered.head())


In [ ]:
def has_only_acgt(sequence):
    encoded = BASE_TO_INDEX[np.frombuffer(sequence.encode("ascii"), dtype=np.uint8)]
    return not np.any(encoded == 255)


def build_sequence_pair_dataset():
    if SEQUENCE_DATASET_PATH.exists() and not RUN_BUILD_DATASET:
        print("Use existing sequence parquet:", SEQUENCE_DATASET_PATH)
        return pd.read_parquet(SEQUENCE_DATASET_PATH)

    df = pd.read_parquet(FILTERED_PATH).copy()
    genome = Fasta(str(FASTA_PATH), as_raw=True, sequence_always_upper=True, rebuild=False)
    rows = []
    skip_counts = Counter()

    for row in tqdm(df.itertuples(index=False), total=len(df), desc=f"Extract REF/ALT {LENGTH}bp"):
        ref = clean_base(row.ReferenceAlleleVCF)
        alt = clean_base(row.AlternateAlleleVCF)
        chrom = str(row.ChromosomeFASTA)
        if chrom not in genome:
            skip_counts["chrom_missing"] += 1
            continue
        pos = int(row.PositionVCF)
        start_1based = pos - FLANK
        end_1based = pos + FLANK
        if start_1based < 1:
            skip_counts["edge_or_short"] += 1
            continue
        ref_seq = str(genome[chrom][start_1based - 1:end_1based]).upper()
        if len(ref_seq) != LENGTH:
            skip_counts["edge_or_short"] += 1
            continue
        if ref_seq[FLANK] != ref:
            skip_counts["ref_mismatch"] += 1
            continue
        if not has_only_acgt(ref_seq):
            skip_counts["non_acgt_context"] += 1
            continue
        alt_seq = ref_seq[:FLANK] + alt + ref_seq[FLANK + 1:]
        record = row._asdict()
        record["ContextStart1Based"] = start_1based
        record["ContextEnd1Based"] = end_1based
        record["ref_seq"] = ref_seq
        record["alt_seq"] = alt_seq
        rows.append(record)

    metadata = pd.DataFrame(rows)
    metadata["Y"] = metadata["Y"].astype(np.int8)
    metadata.to_parquet(SEQUENCE_DATASET_PATH, index=False)
    print("Saved sequence dataset:", SEQUENCE_DATASET_PATH, metadata.shape)
    print("Skipped:", dict(skip_counts))
    print("Label counts:", metadata["Y"].value_counts().sort_index().to_dict())
    assert "LastEvaluated" in metadata.columns
    assert (metadata["ref_seq"].str.len() == LENGTH).all()
    assert (metadata["alt_seq"].str.len() == LENGTH).all()
    sample = metadata.sample(min(20, len(metadata)), random_state=RANDOM_STATE)
    assert all(row.ref_seq[FLANK] == row.ReferenceAlleleVCF for row in sample.itertuples(index=False))
    assert all(row.alt_seq[FLANK] == row.AlternateAlleleVCF for row in sample.itertuples(index=False))
    return metadata

metadata = build_sequence_pair_dataset()
y = metadata["Y"].to_numpy(dtype=np.int8)
REF_SEQS = metadata["ref_seq"].to_numpy(dtype=object)
ALT_SEQS = metadata["alt_seq"].to_numpy(dtype=object)
print("metadata:", metadata.shape)
print("labels:", dict(zip(*np.unique(y, return_counts=True))))
display(metadata[["GeneSymbol", "ClinicalSignificance", "LastEvaluated", "ReviewStatus", "ref_seq", "alt_seq", "Y"]].head())


In [ ]:
def select_label_indices(meta, label_mode):
    if label_mode == "all":
        return np.arange(len(meta), dtype=np.int64)
    if label_mode == "definitive_only":
        clinical = meta["ClinicalSignificance"].fillna("").astype(str).str.strip()
        return np.flatnonzero(clinical.isin(["Benign", "Pathogenic"]).to_numpy()).astype(np.int64)
    raise ValueError(f"Unsupported LABEL_MODE: {label_mode}")


def make_groups(meta):
    groups = meta["GeneSymbol"].fillna("unknown").astype(str).to_numpy(copy=True)
    unknown = (groups == "") | (groups == "unknown") | (groups == "nan")
    row_ids = np.arange(len(groups)).astype(str)
    groups[unknown] = np.char.add("unknown_row_", row_ids[unknown])
    return groups


def make_chromosome_groups(meta):
    groups = meta["Chromosome"].fillna("unknown").astype(str).str.replace("chr", "", case=False, regex=False).to_numpy(copy=True)
    unknown = (groups == "") | (groups == "unknown") | (groups == "nan")
    row_ids = np.arange(len(groups)).astype(str)
    groups[unknown] = np.char.add("unknown_chr_row_", row_ids[unknown])
    return groups


def make_genome_block_groups(meta, chromosome_groups, block_size_bp):
    positions = pd.to_numeric(meta["PositionVCF"], errors="coerce")
    blocks = ((positions - 1) // block_size_bp).astype("Int64")
    chrom = pd.Series(chromosome_groups, index=meta.index).astype(str)
    block_groups = chrom + ":block_" + blocks.astype(str)
    invalid = positions.isna() | chrom.isin(["", "unknown", "nan"]) | blocks.isna()
    if invalid.any():
        row_ids = pd.Series(np.arange(len(meta)), index=meta.index).astype(str)
        block_groups.loc[invalid] = "unknown_block_row_" + row_ids.loc[invalid]
    return block_groups.to_numpy(dtype=str, copy=True)


def make_group_split(labels, groups, random_state):
    all_idx = np.arange(len(labels))
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=random_state)
    train_idx, temp_idx = next(gss1.split(all_idx, labels, groups=groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=random_state)
    val_rel, test_rel = next(gss2.split(temp_idx, labels[temp_idx], groups=groups[temp_idx]))
    return train_idx, temp_idx[val_rel], temp_idx[test_rel]


def maybe_subsample(indices, max_rows, labels, seed):
    indices = np.asarray(indices, dtype=np.int64)
    if max_rows is None or len(indices) <= max_rows:
        return indices
    rng = np.random.default_rng(seed)
    selected = []
    for label in [0, 1]:
        label_idx = indices[labels[indices] == label]
        n_label = int(round(max_rows * len(label_idx) / len(indices)))
        n_label = min(len(label_idx), max(1, n_label))
        selected.append(rng.choice(label_idx, size=n_label, replace=False))
    out = np.concatenate(selected)
    rng.shuffle(out)
    return out.astype(np.int64)


def nearest_distance_to_reference(meta, query_idx, reference_idx):
    ref_pos_by_chr = {}
    for chrom, sub in meta.iloc[reference_idx].groupby("Chromosome", sort=False):
        positions = pd.to_numeric(sub["PositionVCF"], errors="coerce").dropna()
        ref_pos_by_chr[str(chrom)] = np.sort(positions.to_numpy(dtype=np.int64))
    query = meta.iloc[query_idx]
    distances = np.full(len(query_idx), np.inf, dtype=np.float64)
    for i, (chrom, pos) in enumerate(zip(query["Chromosome"].astype(str), pd.to_numeric(query["PositionVCF"], errors="coerce"))):
        if pd.isna(pos):
            continue
        ref_positions = ref_pos_by_chr.get(str(chrom))
        if ref_positions is None or len(ref_positions) == 0:
            continue
        pos_int = int(pos)
        insert_at = np.searchsorted(ref_positions, pos_int)
        best = np.inf
        if insert_at > 0:
            best = min(best, abs(pos_int - int(ref_positions[insert_at - 1])))
        if insert_at < len(ref_positions):
            best = min(best, abs(pos_int - int(ref_positions[insert_at])))
        distances[i] = best
    return distances


def distance_summary(distances):
    finite = distances[np.isfinite(distances)]
    if len(finite) == 0:
        return {"min": float("inf"), "p5": float("inf"), "median": float("inf"), "p95": float("inf")}
    return {"min": float(np.min(finite)), "p5": float(np.percentile(finite, 5)), "median": float(np.median(finite)), "p95": float(np.percentile(finite, 95))}


def variant_keys(meta, indices):
    sub = meta.iloc[np.asarray(indices, dtype=np.int64)]
    return (
        sub["Chromosome"].astype(str) + ":" + sub["PositionVCF"].astype(str) + ":" +
        sub["ReferenceAlleleVCF"].astype(str) + ">" + sub["AlternateAlleleVCF"].astype(str)
    ).to_numpy(dtype=str)


def purge_exact_variant_splits(meta, train_idx, val_idx, test_idx):
    train_keys = set(variant_keys(meta, train_idx))
    keep_val = np.array([key not in train_keys for key in variant_keys(meta, val_idx)], dtype=bool)
    kept_val = val_idx[keep_val]
    reference_keys = train_keys | set(variant_keys(meta, kept_val))
    keep_test = np.array([key not in reference_keys for key in variant_keys(meta, test_idx)], dtype=bool)
    kept_test = test_idx[keep_test]
    return kept_val, kept_test, {
        "enabled": True,
        "val_rows_before_exact_variant_purge": int(len(val_idx)),
        "val_rows_after_exact_variant_purge": int(len(kept_val)),
        "val_rows_purged_by_exact_variant": int((~keep_val).sum()),
        "test_rows_before_exact_variant_purge": int(len(test_idx)),
        "test_rows_after_exact_variant_purge": int(len(kept_test)),
        "test_rows_purged_by_exact_variant": int((~keep_test).sum()),
    }


def purge_nearby_splits(meta, train_idx, val_idx, test_idx, purge_distance_bp):
    val_dist = nearest_distance_to_reference(meta, val_idx, train_idx)
    keep_val = val_dist >= purge_distance_bp
    kept_val = val_idx[keep_val]
    test_ref = np.concatenate([train_idx, kept_val])
    test_dist = nearest_distance_to_reference(meta, test_idx, test_ref)
    keep_test = test_dist >= purge_distance_bp
    kept_test = test_idx[keep_test]
    return kept_val, kept_test, {
        "val_rows_before_purge": int(len(val_idx)),
        "val_rows_after_purge": int(len(kept_val)),
        "val_rows_purged": int((~keep_val).sum()),
        "test_rows_before_purge": int(len(test_idx)),
        "test_rows_after_purge": int(len(kept_test)),
        "test_rows_purged": int((~keep_test).sum()),
        "val_nearest_train_distance_bp": distance_summary(nearest_distance_to_reference(meta, kept_val, train_idx)),
        "test_nearest_train_val_distance_bp": distance_summary(nearest_distance_to_reference(meta, kept_test, test_ref)),
    }


def context_hash(global_idx, mode):
    row = metadata.iloc[int(global_idx)]
    if mode == "exact_ref":
        text = row["ref_seq"]
    elif mode == "exact_tensor":
        text = row["ref_seq"] + ">" + row["alt_seq"]
    else:
        raise ValueError(f"Unsupported sequence purge mode: {mode}")
    return hashlib.blake2b(text.encode("ascii"), digest_size=16).digest()


def purge_sequence_context_splits(train_global, val_global, test_global, mode):
    if mode == "none":
        return val_global, test_global, {"sequence_purge_mode": mode}
    train_hashes = {context_hash(i, mode) for i in tqdm(train_global, desc="hash train contexts", leave=False)}
    keep_val = np.array([context_hash(i, mode) not in train_hashes for i in tqdm(val_global, desc="purge val sequence duplicates", leave=False)])
    kept_val = val_global[keep_val]
    val_hashes = {context_hash(i, mode) for i in tqdm(kept_val, desc="hash kept val contexts", leave=False)}
    reference_hashes = train_hashes | val_hashes
    keep_test = np.array([context_hash(i, mode) not in reference_hashes for i in tqdm(test_global, desc="purge test sequence duplicates", leave=False)])
    kept_test = test_global[keep_test]
    return kept_val, kept_test, {
        "sequence_purge_mode": mode,
        "val_rows_before_sequence_purge": int(len(val_global)),
        "val_rows_after_sequence_purge": int(len(kept_val)),
        "val_rows_purged_by_sequence": int((~keep_val).sum()),
        "test_rows_before_sequence_purge": int(len(test_global)),
        "test_rows_after_sequence_purge": int(len(kept_test)),
        "test_rows_purged_by_sequence": int((~keep_test).sum()),
    }


def label_count_dict(labels, indices):
    values, counts = np.unique(labels[indices], return_counts=True)
    return {str(int(v)): int(c) for v, c in zip(values, counts)}


def prevalence(labels, indices):
    return float(np.mean(labels[indices].astype(np.float64))) if len(indices) else float("nan")


def year_summary(years, indices):
    selected = years.iloc[np.asarray(indices, dtype=np.int64)].dropna().astype(int)
    if len(selected) == 0:
        return {"min": None, "max": None, "counts": {}}
    return {"min": int(selected.min()), "max": int(selected.max()), "counts": {str(int(k)): int(v) for k, v in selected.value_counts().sort_index().items()}}


def split_by_time(years):
    all_idx = np.arange(len(years), dtype=np.int64)
    values = years.to_numpy()
    return (
        all_idx[values <= TEMPORAL_TRAIN_MAX_YEAR],
        all_idx[values == TEMPORAL_VAL_YEAR],
        all_idx[values >= TEMPORAL_TEST_MIN_YEAR],
    )


def split_by_space_time(labels, block_groups, years):
    block_train, block_val, block_test = make_group_split(labels, block_groups, RANDOM_STATE)
    train_blocks = set(block_groups[block_train])
    val_blocks = set(block_groups[block_val])
    test_blocks = set(block_groups[block_test])
    values = years.to_numpy()
    all_idx = np.arange(len(labels), dtype=np.int64)
    return (
        all_idx[np.isin(block_groups, list(train_blocks)) & (values <= TEMPORAL_TRAIN_MAX_YEAR)],
        all_idx[np.isin(block_groups, list(val_blocks)) & (values == TEMPORAL_VAL_YEAR)],
        all_idx[np.isin(block_groups, list(test_blocks)) & (values >= TEMPORAL_TEST_MIN_YEAR)],
    )


def make_split(split_mode, active_idx, meta_for_split, labels):
    years = pd.to_datetime(meta_for_split["LastEvaluated"], errors="coerce").dt.year
    keep = years.notna().to_numpy() if split_mode in {"time_only", "space_time_block", "space_only_matched"} else np.ones(len(meta_for_split), dtype=bool)
    active_idx = active_idx[keep]
    meta_for_split = meta_for_split.loc[keep].reset_index(drop=True)
    labels = labels[keep]
    years = years.loc[keep].reset_index(drop=True)

    chrom_groups = make_chromosome_groups(meta_for_split)
    block_groups = make_genome_block_groups(meta_for_split, chrom_groups, GENOME_BLOCK_SIZE_BP)
    gene_groups = make_groups(meta_for_split)

    if split_mode == "time_only":
        train_local, val_local, test_local = split_by_time(years)
    elif split_mode == "space_time_block":
        train_local, val_local, test_local = split_by_space_time(labels, block_groups, years)
    elif split_mode == "space_only_matched":
        train_local, val_local, test_local = make_group_split(labels, block_groups, RANDOM_STATE)
        target_rows = SPACE_ONLY_TARGET_TRAIN_ROWS
        if target_rows is None and MATCH_SPACE_ONLY_TRAIN_SIZE:
            target_rows = len(split_by_space_time(labels, block_groups, years)[0])
        if target_rows is not None:
            train_local = maybe_subsample(train_local, target_rows, labels, RANDOM_STATE + 17)
    elif split_mode == "genome_block":
        train_local, val_local, test_local = make_group_split(labels, block_groups, RANDOM_STATE)
    elif split_mode in {"gene_group", "purged_gene_group"}:
        train_local, val_local, test_local = make_group_split(labels, gene_groups, RANDOM_STATE)
    else:
        raise ValueError(f"Unsupported split_mode: {split_mode}")

    if min(len(train_local), len(val_local), len(test_local)) == 0:
        raise ValueError(f"Empty split for {split_mode}: train={len(train_local)}, val={len(val_local)}, test={len(test_local)}")

    exact_audit = {"enabled": False}
    if EXACT_VARIANT_PURGE:
        val_local, test_local, exact_audit = purge_exact_variant_splits(meta_for_split, train_local, val_local, test_local)

    coord_audit = {"enabled": False}
    coordinate_purge_modes = {"purged_gene_group", "genome_block", "space_time_block", "space_only_matched"}
    if APPLY_COORDINATE_PURGE_TO_TIME_ONLY:
        coordinate_purge_modes.add("time_only")
    if split_mode in coordinate_purge_modes:
        val_local, test_local, coord_audit = purge_nearby_splits(meta_for_split, train_local, val_local, test_local, PURGE_DISTANCE_BP)

    train_global = active_idx[train_local]
    val_global = active_idx[val_local]
    test_global = active_idx[test_local]
    kept_val_global, kept_test_global, seq_audit = purge_sequence_context_splits(train_global, val_global, test_global, SEQUENCE_PURGE_MODE)
    global_to_local = np.full(len(metadata), -1, dtype=np.int64)
    global_to_local[active_idx] = np.arange(len(active_idx), dtype=np.int64)
    val_local = global_to_local[kept_val_global]
    test_local = global_to_local[kept_test_global]

    train_idx = active_idx[train_local]
    val_idx = active_idx[val_local]
    test_idx = active_idx[test_local]
    train_blocks, val_blocks, test_blocks = set(block_groups[train_local]), set(block_groups[val_local]), set(block_groups[test_local])
    audit = {
        "split_mode": split_mode,
        "genome_block_size_bp": GENOME_BLOCK_SIZE_BP,
        "purge_distance_bp": PURGE_DISTANCE_BP,
        "apply_coordinate_purge_to_time_only": APPLY_COORDINATE_PURGE_TO_TIME_ONLY,
        "sequence_purge_mode": SEQUENCE_PURGE_MODE,
        "temporal_train_max_year": TEMPORAL_TRAIN_MAX_YEAR,
        "temporal_val_year": TEMPORAL_VAL_YEAR,
        "temporal_test_min_year": TEMPORAL_TEST_MIN_YEAR,
        "selected_rows": int(len(active_idx)),
        "train_rows": int(len(train_idx)), "val_rows": int(len(val_idx)), "test_rows": int(len(test_idx)),
        "train_label_counts": label_count_dict(labels, train_local),
        "val_label_counts": label_count_dict(labels, val_local),
        "test_label_counts": label_count_dict(labels, test_local),
        "train_prevalence": prevalence(labels, train_local),
        "val_prevalence": prevalence(labels, val_local),
        "test_prevalence": prevalence(labels, test_local),
        "train_year_summary": year_summary(years, train_local),
        "val_year_summary": year_summary(years, val_local),
        "test_year_summary": year_summary(years, test_local),
        "train_block_count": int(len(train_blocks)), "val_block_count": int(len(val_blocks)), "test_block_count": int(len(test_blocks)),
        "genome_block_overlap_train_val": int(len(train_blocks & val_blocks)),
        "genome_block_overlap_train_test": int(len(train_blocks & test_blocks)),
        "genome_block_overlap_val_test": int(len(val_blocks & test_blocks)),
        "val_nearest_train_distance_bp": distance_summary(nearest_distance_to_reference(meta_for_split, val_local, train_local)),
        "test_nearest_train_val_distance_bp": distance_summary(nearest_distance_to_reference(meta_for_split, test_local, np.concatenate([train_local, val_local]))),
        "exact_variant_purge_audit": exact_audit,
        "coordinate_purge_audit": coord_audit,
        "sequence_purge_audit": seq_audit,
    }
    return train_idx, val_idx, test_idx, audit

active_idx = select_label_indices(metadata, LABEL_MODE)
metadata_for_split = metadata.iloc[active_idx].reset_index(drop=True)
y_for_split = np.asarray(y[active_idx], dtype=np.int8)
print("active rows:", f"{len(active_idx):,}", "labels:", dict(zip(*np.unique(y_for_split, return_counts=True))))


In [ ]:
def resolve_model_path():
    if MODEL_LOCAL_DIR_OVERRIDE:
        p = Path(MODEL_LOCAL_DIR_OVERRIDE)
        if p.exists():
            return str(p)
        raise FileNotFoundError(f"MODEL_LOCAL_DIR_OVERRIDE does not exist: {p}")
    return MODEL_NAME_OR_PATH


def load_tokenizer():
    model_path = resolve_model_path()
    try:
        tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=TRUST_REMOTE_CODE)
    except Exception as exc:
        raise RuntimeError(
            "Cannot load DNABERT-2 tokenizer. On Kaggle, enable internet or add the Hugging Face model "
            "as an input dataset and set MODEL_LOCAL_DIR_OVERRIDE to that directory.\n"
            f"Original error: {type(exc).__name__}: {exc}\n\nTraceback:\n{traceback.format_exc()}"
        ) from exc
    if tok.pad_token is None:
        if tok.eos_token is not None:
            tok.pad_token = tok.eos_token
        elif tok.unk_token is not None:
            tok.pad_token = tok.unk_token
        else:
            tok.add_special_tokens({"pad_token": "[PAD]"})
    return tok


tokenizer = load_tokenizer()
print("tokenizer:", tokenizer.__class__.__name__)
print("pad_token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("model_max_length:", tokenizer.model_max_length)


In [ ]:
class DNABertPairDataset(Dataset):
    def __init__(self, indices, perturbation="none", seed=0):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.perturbation = perturbation
        rng = np.random.default_rng(seed)
        self.source_indices = self.indices.copy()
        if len(self.source_indices) > 1:
            rng.shuffle(self.source_indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = int(self.indices[item])
        ref_seq = str(REF_SEQS[idx])
        alt_seq = str(ALT_SEQS[idx])
        if self.perturbation in {"alt_center_shuffle", "ref_alt_pair_shuffle"}:
            src_idx = int(self.source_indices[item])
            if self.perturbation == "alt_center_shuffle":
                src_alt = str(ALT_SEQS[src_idx])
                alt_seq = alt_seq[:FLANK] + src_alt[FLANK] + alt_seq[FLANK + 1:]
            else:
                ref_seq = str(REF_SEQS[src_idx])
                alt_seq = str(ALT_SEQS[src_idx])
        elif self.perturbation != "none":
            raise ValueError(f"Unknown perturbation: {self.perturbation}")
        return ref_seq, alt_seq, float(y[idx]), idx


def collate_ref_alt_pair(batch):
    ref_seqs, alt_seqs, labels, indices = zip(*batch)
    encoded = tokenizer(
        list(ref_seqs),
        list(alt_seqs),
        padding=True,
        truncation=True,
        max_length=MAX_TOKEN_LENGTH,
        return_token_type_ids=False,
        return_tensors="pt",
    )
    encoded["labels"] = torch.tensor(labels, dtype=torch.float32)
    encoded["indices"] = torch.tensor(indices, dtype=torch.long)
    return encoded


def make_loader(indices, batch_size, shuffle=False, sampler=None, perturbation="none", seed=0):
    kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle if sampler is None else False,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_ref_alt_pair,
    )
    if NUM_WORKERS > 0:
        kwargs.update(prefetch_factor=PREFETCH_FACTOR, persistent_workers=PERSISTENT_WORKERS)
    return DataLoader(DNABertPairDataset(indices, perturbation=perturbation, seed=seed), **kwargs)


def make_weighted_sampler(indices):
    labels = np.asarray(y[indices], dtype=np.int64)
    if IMBALANCE_STRATEGY != "weighted_sampler":
        return None
    counts = np.bincount(labels, minlength=2).astype(np.float64)
    weights = 1.0 / np.maximum(counts, 1.0)
    sample_weights = weights[labels]
    return WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double), num_samples=len(sample_weights), replacement=True)


def compute_pos_weight(indices):
    labels = np.asarray(y[indices], dtype=np.int64)
    pos = max(1, int((labels == 1).sum()))
    neg = max(1, int((labels == 0).sum()))
    if IMBALANCE_STRATEGY == "pos_weight":
        return float(neg / pos)
    return 1.0


In [ ]:
def load_dnabert2_config(model_name_or_path):
    config = AutoConfig.from_pretrained(model_name_or_path, trust_remote_code=TRUST_REMOTE_CODE)
    if not hasattr(config, "pad_token_id") or config.pad_token_id is None:
        pad_token_id = tokenizer.pad_token_id if "tokenizer" in globals() and tokenizer.pad_token_id is not None else 0
        config.pad_token_id = int(pad_token_id)
        print("DNABERT-2 config missing pad_token_id; set from tokenizer:", config.pad_token_id)
    if FORCE_PYTORCH_ATTENTION:
        if hasattr(config, "attention_probs_dropout_prob") and getattr(config, "attention_probs_dropout_prob") == 0:
            config.attention_probs_dropout_prob = PYTORCH_ATTENTION_DROPOUT
            print("FORCE_PYTORCH_ATTENTION: set attention_probs_dropout_prob", config.attention_probs_dropout_prob)
        for attr in ("use_flash_attn", "use_flash_attention", "use_flash_attention_2"):
            if hasattr(config, attr):
                setattr(config, attr, False)
                print(f"FORCE_PYTORCH_ATTENTION: set {attr}=False")
    return config


class DNABert2PairClassifier(nn.Module):
    def __init__(self, model_name_or_path, dropout=0.15):
        super().__init__()
        try:
            config = load_dnabert2_config(model_name_or_path)
            self.backbone = AutoModel.from_pretrained(
                model_name_or_path,
                trust_remote_code=TRUST_REMOTE_CODE,
                config=config,
                low_cpu_mem_usage=LOW_CPU_MEM_USAGE,
            )
        except Exception as exc:
            raise RuntimeError(
                "Cannot load DNABERT-2 model. On Kaggle, enable internet or add the Hugging Face model "
                "as an input dataset and set MODEL_LOCAL_DIR_OVERRIDE to that directory.\n"
                f"Original error: {type(exc).__name__}: {exc}\n\nTraceback:\n{traceback.format_exc()}"
            ) from exc
        hidden_size = getattr(self.backbone.config, "hidden_size", None) or getattr(self.backbone.config, "d_model", None)
        if hidden_size is None:
            raise ValueError("Cannot infer hidden size from backbone config.")
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(int(hidden_size), 1)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state if hasattr(outputs, "last_hidden_state") else outputs[0]
        if attention_mask is None:
            pooled = hidden.mean(dim=1)
        else:
            mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        return self.classifier(self.dropout(pooled)).squeeze(1)


def get_module_by_path(root, path):
    module = root
    for part in path.split("."):
        if not hasattr(module, part):
            return None
        module = getattr(module, part)
    return module


def find_transformer_layers(backbone):
    candidate_paths = [
        "bert.encoder.layer",
        "encoder.layer",
        "encoder.layers",
        "transformer.encoder.layer",
        "transformer.h",
        "model.layers",
        "layers",
    ]
    for path in candidate_paths:
        module = get_module_by_path(backbone, path)
        if isinstance(module, (nn.ModuleList, list, tuple)) and len(module) > 0:
            return list(module), path
    best_layers, best_name = [], ""
    for name, module in backbone.named_modules():
        if isinstance(module, nn.ModuleList) and len(module) > len(best_layers):
            best_layers, best_name = list(module), name
    return best_layers, best_name


def apply_finetune_mode(model):
    for param in model.backbone.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

    if FINETUNE_MODE == "full":
        for param in model.backbone.parameters():
            param.requires_grad = True
    elif FINETUNE_MODE == "last_layers":
        layers, path = find_transformer_layers(model.backbone)
        if not layers:
            print("WARNING: cannot find transformer layers; falling back to head_only.")
        else:
            for layer in layers[-UNFREEZE_LAST_N_LAYERS:]:
                for param in layer.parameters():
                    param.requires_grad = True
            print(f"Unfrozen last {min(UNFREEZE_LAST_N_LAYERS, len(layers))}/{len(layers)} layers from {path}")
    elif FINETUNE_MODE == "head_only":
        pass
    else:
        raise ValueError(f"Unsupported FINETUNE_MODE: {FINETUNE_MODE}")


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return int(total), int(trainable)


def make_model():
    model = DNABert2PairClassifier(resolve_model_path(), dropout=DROPOUT)
    if len(tokenizer) > getattr(model.backbone.config, "vocab_size", len(tokenizer)):
        model.backbone.resize_token_embeddings(len(tokenizer))
    apply_finetune_mode(model)
    total, trainable = count_parameters(model)
    print(f"parameters: total={total:,} trainable={trainable:,} ({trainable / total:.2%})")
    return model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = USE_AMP and DEVICE.type == "cuda"
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("AMP_ENABLED:", AMP_ENABLED)


In [ ]:
def safe_auc(fn, labels, probs):
    try:
        return float(fn(labels, probs))
    except ValueError:
        return float("nan")


def evaluate_probs(labels, probs):
    pred = (probs >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(labels, pred)),
        "roc_auc": safe_auc(roc_auc_score, labels, probs),
        "pr_auc": safe_auc(average_precision_score, labels, probs),
    }


def threshold_table(labels, probs):
    precision, recall, thresholds = precision_recall_curve(labels, probs)
    rows = []
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        f1 = 0.0 if p + r == 0 else 2 * p * r / (p + r)
        f2 = 0.0 if 4 * p + r == 0 else 5 * p * r / (4 * p + r)
        rows.append({"threshold": float(t), "precision": float(p), "recall": float(r), "f1": float(f1), "f2": float(f2)})
    return pd.DataFrame(rows)


def metrics_at_threshold(labels, probs, threshold):
    pred = (probs >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(labels, pred)),
        "precision_pathogenic": float(precision_score(labels, pred, zero_division=0)),
        "recall_pathogenic": float(recall_score(labels, pred, zero_division=0)),
        "f1_pathogenic": float(f1_score(labels, pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(labels, pred).tolist(),
    }


def batch_to_device(batch):
    labels = batch.pop("labels").to(DEVICE, non_blocking=True)
    _ = batch.pop("indices", None)
    model_inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
    return model_inputs, labels


def run_eval(model, loader, criterion, desc="eval"):
    model.eval()
    losses, probs_all, labels_all = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc, leave=False):
            model_inputs, labels = batch_to_device(batch)
            with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
                logits = model(**model_inputs)
                loss = criterion(logits, labels)
            losses.append(float(loss.detach().cpu()) * len(labels))
            probs_all.append(torch.sigmoid(logits.detach()).cpu().numpy())
            labels_all.append(labels.detach().cpu().numpy())
    labels = np.concatenate(labels_all).astype(int)
    probs = np.concatenate(probs_all)
    metrics = evaluate_probs(labels, probs)
    metrics["loss"] = float(np.sum(losses) / len(labels))
    return metrics, probs, labels


def run_train_epoch(model, loader, criterion, optimizer, scheduler, scaler):
    model.train()
    losses = []
    optimizer.zero_grad(set_to_none=True)
    last_lr = float(optimizer.param_groups[0]["lr"])
    last_grad_norm = float("nan")
    for step, batch in enumerate(tqdm(loader, desc="train", leave=False), start=1):
        model_inputs, labels = batch_to_device(batch)
        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            logits = model(**model_inputs)
            loss = criterion(logits, labels) / GRAD_ACCUM_STEPS
        if scaler is not None and scaler.is_enabled():
            scaler.scale(loss).backward()
        else:
            loss.backward()
        losses.append(float(loss.detach().cpu()) * GRAD_ACCUM_STEPS * len(labels))
        if step % GRAD_ACCUM_STEPS == 0 or step == len(loader):
            if scaler is not None and scaler.is_enabled():
                scaler.unscale_(optimizer)
            if GRAD_CLIP_NORM and GRAD_CLIP_NORM > 0:
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                last_grad_norm = float(grad_norm.detach().cpu())
            if scaler is not None and scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            if scheduler is not None:
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            last_lr = float(optimizer.param_groups[0]["lr"])
    return {"loss": float(np.sum(losses) / len(loader.dataset)), "lr": last_lr, "grad_norm": last_grad_norm}


def make_optimizer(model):
    encoder_params = [p for p in model.backbone.parameters() if p.requires_grad]
    classifier_params = [p for p in model.classifier.parameters() if p.requires_grad]
    groups = []
    if encoder_params:
        groups.append({"params": encoder_params, "lr": ENCODER_LR})
    if classifier_params:
        groups.append({"params": classifier_params, "lr": CLASSIFIER_LR})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


def make_scheduler(optimizer, num_training_batches, epochs):
    total_steps = max(1, math.ceil(num_training_batches / GRAD_ACCUM_STEPS) * epochs)
    warmup_steps = int(round(total_steps * WARMUP_RATIO))
    return get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)


In [ ]:
if RUN_SMOKE_TEST:
    split_mode = SPLIT_MODES[0]
    train_idx, val_idx, test_idx, split_audit = make_split(split_mode, active_idx, metadata_for_split, y_for_split)
    smoke_idx = train_idx[:8]
    loader = make_loader(smoke_idx, batch_size=8)
    batch = next(iter(loader))
    print("Smoke split:", split_mode)
    print(json.dumps({k: split_audit[k] for k in ["split_mode", "train_rows", "val_rows", "test_rows", "train_year_summary", "val_year_summary", "test_year_summary", "genome_block_overlap_train_test", "coordinate_purge_audit"]}, indent=2))
    print("tokenized input_ids:", tuple(batch["input_ids"].shape))
    print("attention_mask:", tuple(batch["attention_mask"].shape), "labels:", tuple(batch["labels"].shape))
    if len(tokenizer) > 0:
        decoded = tokenizer.decode(batch["input_ids"][0][:40])
        print("decoded first tokens preview:", decoded[:200])
    smoke_model = make_model().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    model_inputs, labels = batch_to_device(batch)
    logits = smoke_model(**model_inputs)
    loss = criterion(logits, labels)
    loss.backward()
    print("logits:", tuple(logits.shape), "loss:", float(loss.detach().cpu()))
    del smoke_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
def train_one_split(split_mode, debug=False):
    print("\n===== SPLIT", split_mode, "debug=", debug, "=====")
    train_idx, val_idx, test_idx, split_audit = make_split(split_mode, active_idx, metadata_for_split, y_for_split)
    if debug:
        train_idx = maybe_subsample(train_idx, DEBUG_MAX_TRAIN_ROWS, y, RANDOM_STATE)
        val_idx = maybe_subsample(val_idx, DEBUG_MAX_VAL_ROWS, y, RANDOM_STATE + 1)
        test_idx = maybe_subsample(test_idx, DEBUG_MAX_TEST_ROWS, y, RANDOM_STATE + 2)

    run_name = f"{SAFE_BASE}_{split_mode}"
    if debug:
        run_name += "_debug"
    paths = {
        "model": MODEL_DIR / f"clinvar_{run_name}_pytorch.pt",
        "metrics": PROCESSED_DIR / f"{run_name}_metrics.json",
        "history": PROCESSED_DIR / f"{run_name}_history.parquet",
        "predictions": PROCESSED_DIR / f"{run_name}_test_predictions.parquet",
        "thresholds": PROCESSED_DIR / f"{run_name}_thresholds.parquet",
        "val_thresholds": PROCESSED_DIR / f"{run_name}_val_thresholds.parquet",
        "split_indices": PROCESSED_DIR / f"{run_name}_split_indices.npz",
    }

    train_sampler = make_weighted_sampler(train_idx)
    train_loader = make_loader(train_idx, BATCH_SIZE, sampler=train_sampler, shuffle=train_sampler is None)
    val_loader = make_loader(val_idx, BATCH_SIZE)
    test_loader = make_loader(test_idx, BATCH_SIZE)

    pos_weight_value = compute_pos_weight(train_idx)
    criterion = nn.BCEWithLogitsLoss() if pos_weight_value == 1.0 else nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, device=DEVICE))
    model = make_model().to(DEVICE)
    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer, len(train_loader), EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

    history = []
    best_val_pr_auc = -np.inf
    best_state = None
    start = time.time()
    for epoch in range(1, EPOCHS + 1):
        print(f"Epoch {epoch}/{EPOCHS}")
        train_metrics = run_train_epoch(model, train_loader, criterion, optimizer, scheduler, scaler)
        val_metrics, _, _ = run_eval(model, val_loader, criterion, desc="val")
        row = {
            "phase": "main",
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_lr": train_metrics["lr"],
            "train_grad_norm": train_metrics["grad_norm"],
            "val_loss": val_metrics["loss"],
            "val_roc_auc": val_metrics["roc_auc"],
            "val_pr_auc": val_metrics["pr_auc"],
        }
        history.append(row)
        print(row)
        if val_metrics["pr_auc"] > best_val_pr_auc:
            best_val_pr_auc = val_metrics["pr_auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print("New best val_pr_auc:", f"{best_val_pr_auc:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    final_val_metrics, val_probs, val_labels = run_eval(model, val_loader, criterion, desc="final val")
    test_metrics, test_probs, test_labels = run_eval(model, test_loader, criterion, desc="test")
    val_thresholds = threshold_table(val_labels, val_probs)
    test_thresholds = threshold_table(test_labels, test_probs)
    best_val_f1 = val_thresholds.loc[val_thresholds["f1"].idxmax()].to_dict()
    best_val_f2 = val_thresholds.loc[val_thresholds["f2"].idxmax()].to_dict()
    val_prev = float(np.mean(val_labels.astype(np.float64)))
    test_prev = float(np.mean(test_labels.astype(np.float64)))

    predictions = metadata.iloc[test_idx].copy()
    predictions["y_true"] = test_labels
    predictions["pred_proba_pathogenic"] = test_probs
    predictions["pred_label_best_val_f1"] = (test_probs >= float(best_val_f1["threshold"])).astype(int)
    predictions["pred_label_best_val_f2"] = (test_probs >= float(best_val_f2["threshold"])).astype(int)

    metrics = {
        "model_name": run_name,
        "input_design": "dnabert2_ref_alt_pair",
        "model_name_or_path": MODEL_NAME_OR_PATH,
        "length": LENGTH,
        "max_token_length": MAX_TOKEN_LENGTH,
        "finetune_mode": FINETUNE_MODE,
        "unfreeze_last_n_layers": UNFREEZE_LAST_N_LAYERS,
        "split_audit": split_audit,
        "best_val_pr_auc": float(best_val_pr_auc),
        "final_val_metrics": final_val_metrics,
        "test_metrics": test_metrics,
        "val_prevalence": val_prev,
        "test_prevalence": test_prev,
        "val_pr_auc_lift": float(final_val_metrics["pr_auc"] / val_prev) if val_prev > 0 else float("nan"),
        "test_pr_auc_lift": float(test_metrics["pr_auc"] / test_prev) if test_prev > 0 else float("nan"),
        "best_val_f1_threshold": {k: float(v) for k, v in best_val_f1.items()},
        "best_val_f2_threshold": {k: float(v) for k, v in best_val_f2.items()},
        "test_at_best_val_f1_threshold": metrics_at_threshold(test_labels, test_probs, float(best_val_f1["threshold"])),
        "test_at_best_val_f2_threshold": metrics_at_threshold(test_labels, test_probs, float(best_val_f2["threshold"])),
        "train_rows": int(len(train_idx)), "val_rows": int(len(val_idx)), "test_rows": int(len(test_idx)),
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "encoder_lr": ENCODER_LR,
        "classifier_lr": CLASSIFIER_LR,
        "imbalance_strategy": IMBALANCE_STRATEGY,
        "pos_weight": float(pos_weight_value),
        "runtime_minutes": float((time.time() - start) / 60.0),
    }
    pd.DataFrame(history).to_parquet(paths["history"], index=False)
    predictions.to_parquet(paths["predictions"], index=False)
    test_thresholds.to_parquet(paths["thresholds"], index=False)
    val_thresholds.to_parquet(paths["val_thresholds"], index=False)
    np.savez_compressed(paths["split_indices"], train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)
    paths["metrics"].write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    torch.save({"model_state_dict": model.state_dict(), "config": metrics}, paths["model"])
    print("Saved metrics:", paths["metrics"])
    print("Saved model:", paths["model"])
    return metrics

if RUN_MINI_TRAIN:
    _ = train_one_split(SPLIT_MODES[0], debug=True)


In [ ]:
summaries = []
if RUN_FULL_TRAIN:
    for split_mode in SPLIT_MODES:
        summaries.append(train_one_split(split_mode, debug=False))

if summaries:
    summary_df = pd.DataFrame([
        {
            "model_name": m["model_name"],
            "split_mode": m["split_audit"]["split_mode"],
            "train_rows": m["train_rows"],
            "val_rows": m["val_rows"],
            "test_rows": m["test_rows"],
            "test_prevalence": m["test_prevalence"],
            "test_pr_auc": m["test_metrics"]["pr_auc"],
            "test_pr_auc_lift": m["test_pr_auc_lift"],
            "test_roc_auc": m["test_metrics"]["roc_auc"],
            "test_f1_at_best_val_f1": m["test_at_best_val_f1_threshold"]["f1_pathogenic"],
            "precision_at_best_val_f1": m["test_at_best_val_f1_threshold"]["precision_pathogenic"],
            "recall_at_best_val_f1": m["test_at_best_val_f1_threshold"]["recall_pathogenic"],
        }
        for m in summaries
    ])
    display(summary_df)
    summary_path = PROCESSED_DIR / f"{SAFE_BASE}_diagnostic_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print("Saved summary:", summary_path)
else:
    print("RUN_FULL_TRAIN=False, no full summary generated.")


## Negative controls

Chạy sau full train nếu muốn kiểm tra model có dùng tín hiệu biến thể thật không.

- `alt_center_shuffle`: giữ REF/background, tráo ALT center.
- `ref_alt_pair_shuffle`: thay cả REF/ALT pair bằng pair của sample khác nhưng giữ label gốc.
- `label_shuffle_by_block`: giữ score gốc, tráo label trong cùng genome block.


In [ ]:
def make_saved_run_paths(split_mode, debug=False):
    run_name = f"{SAFE_BASE}_{split_mode}"
    if debug:
        run_name += "_debug"
    return run_name, {
        "model": MODEL_DIR / f"clinvar_{run_name}_pytorch.pt",
        "metrics": PROCESSED_DIR / f"{run_name}_metrics.json",
        "split_indices": PROCESSED_DIR / f"{run_name}_split_indices.npz",
        "negative_controls": PROCESSED_DIR / f"{run_name}_negative_control_summary.parquet",
    }


def stratified_subsample_global_indices(indices, max_rows, seed):
    indices = np.asarray(indices, dtype=np.int64)
    if max_rows is None or len(indices) <= max_rows:
        return indices
    rng = np.random.default_rng(seed)
    labels = np.asarray(y[indices], dtype=np.int8)
    selected = []
    for label in [0, 1]:
        label_idx = indices[labels == label]
        n_label = int(round(max_rows * len(label_idx) / len(indices)))
        n_label = min(len(label_idx), max(1, n_label))
        selected.append(rng.choice(label_idx, size=n_label, replace=False))
    out = np.concatenate(selected)
    rng.shuffle(out)
    return out.astype(np.int64)


def make_global_genome_block_keys(meta, indices, block_size_bp):
    sub = meta.iloc[np.asarray(indices, dtype=np.int64)]
    chrom = sub["Chromosome"].fillna("unknown").astype(str).str.replace("chr", "", case=False, regex=False)
    pos = pd.to_numeric(sub["PositionVCF"], errors="coerce")
    block = ((pos - 1) // block_size_bp).astype("Int64")
    keys = chrom + ":block_" + block.astype(str)
    keys = keys.mask(pos.isna(), "unknown_block")
    return keys.to_numpy(dtype=str, copy=True)


def shuffle_labels_within_groups(labels, groups, seed):
    labels = np.asarray(labels, dtype=np.int8).copy()
    groups = np.asarray(groups, dtype=str)
    rng = np.random.default_rng(seed)
    shuffled = labels.copy()
    for group in np.unique(groups):
        loc = np.flatnonzero(groups == group)
        if len(loc) > 1:
            shuffled[loc] = rng.permutation(shuffled[loc])
    return shuffled


def summarize_control_result(name, labels, probs, threshold):
    labels = np.asarray(labels, dtype=np.int8)
    probs = np.asarray(probs, dtype=np.float64)
    prevalence_value = float(labels.mean()) if len(labels) else float("nan")
    pr_auc = safe_auc(average_precision_score, labels, probs)
    row = {
        "control": name,
        "rows": int(len(labels)),
        "prevalence": prevalence_value,
        "roc_auc": safe_auc(roc_auc_score, labels, probs),
        "pr_auc": pr_auc,
        "pr_auc_lift": float(pr_auc / prevalence_value) if prevalence_value > 0 else float("nan"),
    }
    row.update({f"at_val_f1_{k}": v for k, v in metrics_at_threshold(labels, probs, threshold).items()})
    return row


def load_saved_model_for_split(split_mode, debug=False):
    run_name, run_paths = make_saved_run_paths(split_mode, debug=debug)
    if not run_paths["model"].exists():
        raise FileNotFoundError(f"Missing model checkpoint for {split_mode}: {run_paths['model']}")
    try:
        checkpoint = torch.load(run_paths["model"], map_location=DEVICE, weights_only=False)
    except TypeError:
        checkpoint = torch.load(run_paths["model"], map_location=DEVICE)
    model = make_model().to(DEVICE)
    state = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
    model.load_state_dict(state)
    model.eval()
    return run_name, run_paths, model


def evaluate_input_perturbation(model, indices, perturbation, threshold, seed):
    loader = make_loader(indices, NEGATIVE_CONTROL_BATCH_SIZE, perturbation=perturbation, seed=seed)
    criterion = nn.BCEWithLogitsLoss()
    _, probs, labels = run_eval(model, loader, criterion, desc=f"negative control {perturbation}")
    return summarize_control_result(perturbation, labels, probs, threshold), probs, labels


def run_negative_controls_for_split(split_mode, debug=False):
    seed = RANDOM_STATE + 2027
    run_name, run_paths, model = load_saved_model_for_split(split_mode, debug=debug)
    if not run_paths["split_indices"].exists():
        raise FileNotFoundError(f"Missing split indices for {split_mode}: {run_paths['split_indices']}")
    if not run_paths["metrics"].exists():
        raise FileNotFoundError(f"Missing metrics JSON for {split_mode}: {run_paths['metrics']}")
    with np.load(run_paths["split_indices"]) as split_data:
        test_indices = np.asarray(split_data["test_idx"], dtype=np.int64)
    with open(run_paths["metrics"], "r", encoding="utf-8") as f:
        saved_metrics = json.load(f)
    threshold = float(saved_metrics["best_val_f1_threshold"]["threshold"])
    test_indices = stratified_subsample_global_indices(test_indices, NEGATIVE_CONTROL_MAX_ROWS, seed)

    print("\n===== Negative controls", split_mode, "=====")
    print("rows:", f"{len(test_indices):,}", "threshold:", threshold)
    rows = []
    original_row, original_probs, original_labels = evaluate_input_perturbation(model, test_indices, "none", threshold, seed)
    original_row["note"] = "original input on same diagnostic subset"
    rows.append(original_row)
    for perturbation in ["alt_center_shuffle", "ref_alt_pair_shuffle"]:
        row, _, _ = evaluate_input_perturbation(model, test_indices, perturbation, threshold, seed)
        row["note"] = "same labels, corrupted REF/ALT input"
        rows.append(row)
    block_keys = make_global_genome_block_keys(metadata, test_indices, GENOME_BLOCK_SIZE_BP)
    shuffled_labels = shuffle_labels_within_groups(original_labels, block_keys, seed)
    row = summarize_control_result("label_shuffle_by_block", shuffled_labels, original_probs, threshold)
    row["note"] = "same original scores, labels shuffled within genome block"
    rows.append(row)
    out = pd.DataFrame(rows)
    out.insert(0, "split_mode", split_mode)
    out.insert(0, "model_name", run_name)
    out.to_parquet(run_paths["negative_controls"], index=False)
    print("Saved:", run_paths["negative_controls"])
    return out

negative_control_summaries = []
if RUN_NEGATIVE_CONTROLS:
    for split_mode in SPLIT_MODES:
        try:
            negative_control_summaries.append(run_negative_controls_for_split(split_mode, debug=False))
        except FileNotFoundError as exc:
            print("Skip negative controls:", exc)
    if negative_control_summaries:
        negative_control_df = pd.concat(negative_control_summaries, ignore_index=True)
        display_cols = [
            "split_mode", "control", "rows", "prevalence", "pr_auc", "pr_auc_lift", "roc_auc",
            "at_val_f1_f1_pathogenic", "at_val_f1_precision_pathogenic", "at_val_f1_recall_pathogenic", "note",
        ]
        display(negative_control_df[display_cols])
        combined_path = PROCESSED_DIR / f"{SAFE_BASE}_negative_control_summary.csv"
        negative_control_df.to_csv(combined_path, index=False)
        print("Saved combined summary:", combined_path)
else:
    print("RUN_NEGATIVE_CONTROLS=False, skip negative-control diagnostics.")


## Cách đọc kết quả

Chạy cùng model dưới 3 mode để chẩn đoán leakage:

```python
SPLIT_MODES = ["space_only_matched", "time_only", "space_time_block"]
```

- `space_only_matched`: kiểm soát không gian, train size khớp với temporal strict.
- `time_only`: kiểm soát thời gian nhưng cho phép gene/block overlap; mặc định không coordinate purge 5kb.
- `space_time_block`: strict nhất, tách cả năm và vùng genome.

Caveat: đây vẫn là pseudo-temporal vì `LastEvaluated` không phải ngày variant xuất hiện lần đầu. True prospective validation cần diff các snapshot ClinVar theo thời gian.
